### Webscraper for extracting newly published names, strains and accession numbers from the weeekly IJSEM email (saved in html). Script then compares the IJSEM names to the NCBI names and generates a report used for taxonomy updates. 
### Uses Natural Language processing combined with REGEX to extract information from text
### Notes on selenium. There is a lot of bad advise in stack overflow that refers to old versions of selenium. Need to refer to the offocial documentation https://selenium-python.readthedocs.io/ If you run out of space in your linux directory the script will crash. There is a cache file written by selenium that may then need to be deleted to get the chrome driver working again. Follow the path in the error message and delete that cache directory. Selenium needs a webdriver installed, find details at https://sites.google.com/chromium.org/driver/downloads

### NOTE selenium cache files are quite large and need to be cleaned up periodically

In [ ]:
pip install "numpy<2.0" 

In [ ]:
pip install spacy
#pip install -U pip setuptools wheel
#pip install -U spacy
#pip install --upgrade spacy

### Install the model at the command lines
small model
python -m spacy download en_core_web_sm

medium model
python -m spacy download en_core_web_md

large model
python -m spacy download en_core_web_lg

### create training set https://spacy.io/usage/training

Label-studio works the best for labelling data. Export as .json when finished, they use labelstudio_to_spacy2.py to convert the .json files to .spacy file for training. 

https://spacy.io/usage/training#quickstart

python -m spacy init fill-config ./base_config.cfg ./config.cfg

This NER annotator worked better than the one above. Used the DocBin technique to convert to .spacy file. Spacy training command with config.cfg then worked.

Config.cfg needs to be created for the spacy command line training. 

Follow the instructions in the quickstart for the base_config.cfg file. Selected OS and clicked NER

https://spacy.io/usage/training#quickstart

Run the following at command line to create the config.cfg. 

python -m spacy init fill-config ./base_config.cfg ./config.cfg

Once the config file was done and spacy file was created run the following at command line to train the model

To debug the data:
python -m spacy debug data config.cfg --paths.train ./train.spacy --paths.dev ./train.spacy

To debug the config file file:
python -m spacy train config.cfg --output ./output

or

python -m spacy train config.cfg 

train the model at command line
python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./train.spacy

creates two directories model-best and model-last. I used model-best 

nlp1 = spacy.load(r"./output/model-best")


Notes
python -m spacy init fill-config ./base_config.cfg ./config.cfg

python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./dev.spacy

https://spacy.io/usage/training#basics

### Train spacy if not already done. Using https://labelstud.io/


In [1]:
from spacy.tokens import DocBin
import pandas as pd
import json
import os

### train the model at command line
python -m spacy train config3.cfg --output ./output --paths.train ./train.spacy --paths.dev ./dev.spacy 

### Main webscraper program start


In [2]:
import selenium
import tempfile
from selenium import webdriver
from selenium.webdriver.common.by import By
chrome_path = "/home/mcveigh/.cache/selenium/chrome/linux64/138.0.7204.183/chrome"
from selenium.webdriver.chrome.options import Options

import time
import IPython
import pandas as pd
import re
import os
import sys
import bs4
from bs4 import BeautifulSoup
import requests
import numpy as np
from datetime import datetime
from nameparser import HumanName
import spacy
#nlp = spacy.load("en_core_web_sm")
nlp = spacy.load("en_core_web_md")
from spacy.matcher import PhraseMatcher
from spacy import displacy

In [18]:
#test block for testing displacy
text2 = ("The type strain is 2205BS29-5T (=LMG 33062T =KACC 23240T), which is isolated from a marine sponge, P. elegans, "
"collected from Beomseom in Jeju-Island, Republic of Korea."

"The DNA G+C content of strain 2205BS29-5T is 67.8%. The GenBank accession numbers for the 16S rRNA gene and" 
"whole-genome sequences of strain 2205BS29-5T are OQ569368 and JAVAMQ000000000, respectively.")

In [2]:
#test but need this later
import IPython
nlp1 = spacy.load(r"./output/model-best") #load the best model
doc = nlp1(text2)
from IPython.display import display
from spacy import displacy
from IPython.core.display import HTML, display
displacy.render(doc, style="ent")

NameError: name 'text2' is not defined

### Find URLS from saved email in html - save as from outlook in htm format. URLs are extracted and saved as input for selenium

In [3]:
input = (r'IJSEMemail83.htm')
output = (r'NameCheckweek83.xlsx')
alldescriptions = (r'all_descriptions83')

In [ ]:
#Only need this if you suspect a problem with beautiful soup not imporing the URL list correctly. Otherwise skip this step
from bs4.diagnose import diagnose
with open (input, encoding = 'unicode_escape') as f:
    data = f.read()
diagnose(data)

#### alternative Beautifiul soup code for extracting URLS with autodetect encoding

In [4]:
#alternative Beautifiul soup with autodetect encoding
from charset_normalizer import from_path

# Auto-detect file encoding
result = from_path(input).best()
html = str(result)

# Parse with BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
text = soup.get_text()

# Extract all http/https links
urls = re.findall(r"https?://\S+", text)
urls = [url.rstrip('.,);') for url in urls]

# Filter links as needed
filtered_urls = [
    url for url in urls
    if all(exclude not in url for exclude in ["TandC", "myaccount", "join-the-society"])
    # Remove "doi.org" from filter if you want article links
]

if not filtered_urls:
    raise ValueError("No usable URLs found!")

print(f"Found {len(filtered_urls)} URLs:")
print("\n".join(filtered_urls))


Found 9 URLs:
http://dx.doi.org/10.1099/ijsem.0.007071
http://dx.doi.org/10.1099/ijsem.0.007084
http://dx.doi.org/10.1099/ijsem.0.007081
http://dx.doi.org/10.1099/ijsem.0.007075
http://dx.doi.org/10.1099/ijsem.0.007082
http://dx.doi.org/10.1099/ijsem.0.007083
http://dx.doi.org/10.1099/ijsem.0.007069
http://dx.doi.org/10.1099/ijsem.0.007076
http://dx.doi.org/10.1099/ijsem.0.007079


In [5]:
ALLOWED_STRAIN_LABELS = {"strain"}

def find_strains(description):
    """
    Find strain entities using the trained spaCy NER model, but ONLY return
    entities whose label is in ALLOWED_STRAIN_LABELS.
    Returns a de-duplicated list of strain strings.
    """
    if not description:
        return []

    results = []
    doc = nlp(description)

    for sent in doc.sents:
        # Only process sentences containing strain keywords (keeps your original intent)
        # We run matcher on the sentence text for a cheap filter.
        if not phrase_matcher(nlp(sent.text)):
            continue

        doc2 = nlp_strain(sent.text)  # run trained model on the sentence

        for ent in doc2.ents:
            if ent.label_ in ALLOWED_STRAIN_LABELS:
                val = ent.text.strip()
                # preserve your original non-ascii cleaning intent
                val = val.encode('ascii', 'ignore').decode('utf-8', errors='ignore').strip()
                if val and val not in results:
                    results.append(val)

    return results


def remove_non_ascii(text):
    """Remove non-ASCII characters"""
    return ''.join(char for char in text if ord(char) < 128)

### Main body - Selenium to extract data from html
### https://nameparser.readthedocs.io/en/latest/index.html to find the last name

In [6]:
from selenium.webdriver.common.by import By
import tempfile
pub_df = pd.DataFrame(columns=['PublishedName', 'Accessions', 'Strains', 'Authority', 'DOI', 'filtered_url'])
pd.set_option('display.max_columns', None)
combined_description = []

for filtered_url in filtered_urls:
    temp_profile = tempfile.mkdtemp()

    options = Options()
    options.binary_location = chrome_path
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)

    counter = 1
    strains = []
    accessions = []
    orgname = []
    doi = []
    author = []
    date = []
    year = []
    author_raw = []
    author = []
    name1 = []
    name2 = []
    author_count = []
    authority = []
    description = None
    description1 = []
    description2 = []
    snumber = []

    #Navigate to the webpage
    driver.get(filtered_url)

    #Allow time for dynamic content to load (you may need to use WebDriverWait for more robust waiting)
    time.sleep(3)
    
    html = driver.page_source
    
    for element in driver.find_elements(By.CLASS_NAME, "item-meta-data__item-title"):
        #print(element.text)
        title = element.text
        #print(title)
        
    for element in driver.find_elements(By.PARTIAL_LINK_TEXT, "doi.org"):
        doi = element.text
        #print(doi)
    
    #find publication year
    for element in driver.find_elements(By.XPATH, "//*[@id='bellowheadercontainer']/main/div[2]/div/ul/li[3]/span/span[2]"):
        date = element.text
        year = date[-4:]
        #print(year)
    
    #find authors
    for element in driver.find_elements(By.XPATH, "//*[@id='bellowheadercontainer']/main/div[2]/div/ul/li[1]/span"):
        author_raw = element.text
        #print(author_raw)
        author = re.sub(r"[\d+]+",'',author_raw)
        author = re.sub(r"†",'',author)
        author = re.sub(r" and ",',',author)
        author = re.sub(r",,",',',author)  
        author = author.split(',')
        author[:] = [item for item in author if item != '']
        #print(author)
        author_count = len(author)
        #print(author_count)
        if author_count == 1:
            name1 = (str(author[0])) 
            name1 = HumanName(name1)
            authority = name1.last + ' ' +str(year)
            #print(authority)
        elif author_count ==2:
            name1 = (str(author[0]))
            name1 = HumanName(name1)
            name2 = (str(author[1]))
            name2 = HumanName(name2)
            authority = name1.last + ' and ' + name2.last + ' ' +str(year)
            #print(authority)
        else:
            name1 = (str(author[0]))
            name1 = HumanName(name1)
            authority = name1.last + ' et al. ' +str(year)
            #print(authority)
    
    #extract data from each species description
    for element in driver.find_elements(By.CSS_SELECTOR, "div.tl-main-part.title"): #finds section headers
        #print(element.text)
        counter += 1
        description = element.text
        #print(description)
        if "Description of" in description:
            strains = []
            #print('found', description)  
            #snumber = 's' + str(counter - 4) + '/p[3]'
            snumber = 's' + str(counter - 4)
            #print('snumber is', snumber)
            for element in driver.find_elements(By.ID, snumber):
                description = element.text
                cleaned_text = remove_non_ascii(description)
                combined_description.append(cleaned_text)
                #print(cleaned_text)
                #print(description)
                        
                #find the organism names 
                orgname = []
                match = [r'(\S+\s+){2}(?=sp. nov)', r'(\S+\s+){2}(?=nom. nov.)', r'(\S+\s+){2}(?=SP. NOV.)', r'(\S+\s+){4}(?=subsp. nov.)']
                regex = re.compile(r'\b(' + '|'.join(match) + r')\b')
                if description is not None:
                    orgname = [m.group() for m in regex.finditer(description)]
                    #print('orgname', orgname)

                #find the accessions
                pattern = [r'[A-Z]{2}\d{6}', r'[A-Z]{4}\d{8}', r'([A-Z]+)(_[A-Z]+)\d{6}', r'[A-Z]{6}\d{9}']
                regex = re.compile(r'\b(' + '|'.join(pattern) + r')\b')
                if description is not None:
                    accessions = [m.group() for m in regex.finditer(description)]
                    #print('accessions', accessions)
    
                #find the strains
                if description is not None:
                    find_strains(cleaned_text)
                    #print('strain names', strains)
    
                #load data into pandas dataframe
                row_data = [orgname, accessions, strains, authority, doi, filtered_url]
                length = len(pub_df)
                pub_df.loc[length] = row_data
            #print('BREAK1')

   
    for element in driver.find_elements(By.CLASS_NAME, "tl-lowest-section"): #finds section headers
        description1 = element.text
        outer_html = element.get_attribute("outerHTML")
        if "Description of" in description1:          
            #print(outer_html)
            spans = soup.findAll('span', attrs = {'class' : 'tl-lowest-section'})
            for span in spans:
                if "Description of" in span.text:   
                    #print (span.text)
                    outer_div_id = span.find_parent('div').get('id')
                    #print(f"Outer div ID: {outer_div_id}, Text: {span.text}")
                    for element in driver.find_elements(By.ID, outer_div_id):
                        description = element.text
                        cleaned_text = remove_non_ascii(description)
                        combined_description.append(cleaned_text)
                        #print(cleaned_text)
                        #print(description)
                        
                    #find the organism names   
                    orgname = []
                    match = [r'(\S+\s+){2}(?=sp. nov)', r'(\S+\s+){2}(?=nom. nov.)', r'(\S+\s+){2}(?=SP. NOV.)', r'(\S+\s+){4}(?=subsp. nov.)']
                    regex = re.compile(r'\b(' + '|'.join(match) + r')\b')
                    if description is not None:
                        orgname = [m.group() for m in regex.finditer(description)]
                        #print('orgname', orgname)

                    #find the accessions
                    pattern = [r'[A-Z]{2}\d{6}', r'[A-Z]{4}\d{8}', r'([A-Z]+)(_[A-Z]+)\d{6}', r'[A-Z]{6}\d{9}']
                    regex = re.compile(r'\b(' + '|'.join(pattern) + r')\b')
                    if description is not None:
                        accessions = [m.group() for m in regex.finditer(description)]
                        #print('accessions', accessions)
    
                    #find the strains
                    strains = []
                    if description is not None:
                        find_strains(cleaned_text)
                        #print('strain names', strains)
    
                    #load data into pandas dataframe
                    row_data = [orgname, accessions, strains, authority, doi, filtered_url]
                    length = len(pub_df)
                    pub_df.loc[length] = row_data
                    #print('BREAK2')

        
#Close the browser window
    driver.quit()    

NameError: name 'phrase_matcher' is not defined

In [6]:
#optional write description to a file
#print(combined_description)
file = open(alldescriptions, "w")
file.writelines(combined_description)
file.close()

In [7]:
pd.set_option('max_colwidth', None)
pub_df['Strains'] = [', '.join(map(str, l)) for l in pub_df['Strains']]
#pub_df['Strains'] = pub_df['Strains'].astype(str) 
pub_df['Strains'] = pub_df['Strains'].astype(pd.StringDtype())
pub_df['Strains'] = pub_df['Strains'].str.replace(',', ', ')
pub_df = pub_df.drop_duplicates(subset='PublishedName', keep="first")
#try drop duplicate accessions here
pub_df.explode(['PublishedName']).reset_index(drop=True)

,PublishedName,Accessions,Strains,Authority,DOI,filtered_url
0,Deinococcus pantiae,"[JBJGDW000000000, PP658428]","SM5_A1T, JCM 36669T, KCTC 43670T",Jiya et al. 2026,https://doi.org/10.1099/ijsem.0.007071,http://dx.doi.org/10.1099/ijsem.0.007071
1,NaN,[],,Fisher et al. 2026,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084
2,Halotolerantifilum yawarlongkerapense,"[PX275517, CP199719]","SD5T, DSM 117693T, ATCC TSD-463T",Fisher et al. 2026,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084
3,Halosubterraneus shenae,"[OR734406, PQ096007, JBMWNX000000000, PP425729, PV600743, JBNLVO000000000]","AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",Ding et al. 2026,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
4,Pseudomonas stagnisoli,[PV166458],"P3C3T, CCM 9468T, CECT 31230T, DSM 119661T",Hasan et al. 2026,https://doi.org/10.1099/ijsem.0.007075,http://dx.doi.org/10.1099/ijsem.0.007075
5,Pseudomonas cimarronensis,[PV166459],"MAC6T, CCM 9470T, CCUG 78313T, CECT 31231T",Hasan et al. 2026,https://doi.org/10.1099/ijsem.0.007075,http://dx.doi.org/10.1099/ijsem.0.007075
6,Pectobacterium sinaloense,"[PV590475, SAMN48267889, CP195798]","LFLA-215T, NCCB 101086T, CMCNRG 1201T, ATCC TSD-577",Valdez-López et al. 2026,https://doi.org/10.1099/ijsem.0.007076,http://dx.doi.org/10.1099/ijsem.0.007076
7,Serinicoccus shuyuelongi,"[PQ579907, PQ579904, JBJFZU000000000, JBJFZT000000000]","LYQ92T, GDMCC 1.5391T, KCTC 59547T, LYQ131, PQ579907, PQ579904, JBJFZU000000000, JBJFZT000000000",Liu et al. 2026,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079
8,Ornithinimicrobium jinqii,"[PQ579903, PQ579902, JBJFZS000000000, JBJFZR000000000]","LYQ121T, GDMCC 1.5402T, KCTC 59549T, LYQ103, PQ579903, PQ579902, JBJFZS000000000, JBJFZR000000000",Liu et al. 2026,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079


In [ ]:
#pub_df['Strains'] = pub_df['Strains'].astype(pd.StringDtype())
#pub_df['Strains'] = pub_df['Strains'].str.replace(',', ', ')
#pub_df

In [8]:
pub2_df = pub_df.explode(['Accessions']).reset_index(drop=True)
#pub2_df
pub4_df = pub2_df.explode(['PublishedName']).reset_index(drop=True)
pub4_df.rename(columns={'Accessions' : 'accession'}, inplace=True)
#pub4_df = pub4_df.dropna()
#pub4_df = pub4_df.drop_duplicates(subset='accession', keep="first")
pub4_df=pub4_df[pub4_df['accession'].isnull() | ~pub4_df[pub4_df['accession'].notnull()].duplicated(subset='accession',keep='first')]
#pub4_df

In [9]:
df_unique= pub4_df.drop_duplicates(["accession"], keep="first")
#df_unique = df_unique.dropna()
#df_unique

### Create a dataframe of unique accessions and look up NCBI taxonomy information of each accession with srcchk

In [ ]:
#df_unique.dtypes

In [10]:
#df_unique['accession'] = df_unique['accession'].astype('str') 
df_unique.loc[:, 'accession'] = df_unique['accession'].astype('str') 

In [11]:
with open('acclist', 'w') as f:
    for text in df_unique['accession'].tolist():
        f.write(text + '\n')

In [12]:
os.system("/netopt/ncbi_tools64/bin/srcchk -i acclist -f taxname,taxid,strain -o acclist.taxdata")


Error: Malformatted ID "nan"


0

In [13]:
taxdata_file_name = (r'acclist.taxdata')    
srcchk_df = pd.read_csv(taxdata_file_name, sep='\t', index_col=None, low_memory=False)
srcchk_df.drop(columns=['Unnamed: 4'], inplace=True)
srcchk_df.rename(columns={'organism' : 'NCBIname'}, inplace=True)
srcchk_df['accession'] = srcchk_df['accession'].astype(str).replace('\.\d+', '', regex=True).astype(str)
srcchk_df = srcchk_df.dropna(subset=['NCBIname'])
srcchk_df 

,accession,NCBIname,taxid,strain
0,JBJGDW000000000,Deinococcus sp. SM5_A1,3379094.0,SM5_A1
1,PP658428,Deinococcus sp.,47478.0,SM5_A1
3,PX275517,Eubacteriales bacterium SD5,3459703.0,SD5
4,CP199719,Eubacteriales bacterium SD5,3459703.0,SD5
5,OR734406,Haloparvum sp.,1963352.0,AD34
6,PQ096007,Haloparvum sp.,1963352.0,AD34
7,JBMWNX000000000,Haloparvum sp. AD34,3234950.0,AD34
8,PP425729,Haloparvum sp.,1963352.0,PAK95
9,PV600743,Haloparvum sp.,1963352.0,PAK95
10,JBNLVO000000000,Haloparvum sp. PAK95,3418962.0,PAK95


### Combine dataframes into one

In [14]:
combine_df=pd.merge(left=pub4_df, right=srcchk_df, left_on='accession', right_on='accession', how = 'outer')
combine_df = combine_df[['PublishedName', 'NCBIname', 'Strains', 'accession', 'strain', 'Authority', 'taxid', 'DOI', 'filtered_url' ]]
combine_df

,PublishedName,NCBIname,Strains,accession,strain,Authority,taxid,DOI,filtered_url
0,Pectobacterium sinaloense,Pectobacterium sinaloense,"LFLA-215T, NCCB 101086T, CMCNRG 1201T, ATCC TSD-577",CP195798,LFLA-215,Valdez-López et al. 2026,3419008.0,https://doi.org/10.1099/ijsem.0.007076,http://dx.doi.org/10.1099/ijsem.0.007076
1,Halotolerantifilum yawarlongkerapense,Eubacteriales bacterium SD5,"SD5T, DSM 117693T, ATCC TSD-463T",CP199719,SD5,Fisher et al. 2026,3459703.0,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084
2,Ornithinimicrobium jinqii,Ornithinimicrobium sp. LYQ103,"LYQ121T, GDMCC 1.5402T, KCTC 59549T, LYQ103, PQ579903, PQ579902, JBJFZS000000000, JBJFZR000000000",JBJFZR000000000,LYQ103,Liu et al. 2026,3378796.0,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079
3,Ornithinimicrobium jinqii,Ornithinimicrobium sp. LYQ121,"LYQ121T, GDMCC 1.5402T, KCTC 59549T, LYQ103, PQ579903, PQ579902, JBJFZS000000000, JBJFZR000000000",JBJFZS000000000,LYQ121,Liu et al. 2026,3378801.0,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079
4,Serinicoccus shuyuelongi,Serinicoccus sp. LYQ131,"LYQ92T, GDMCC 1.5391T, KCTC 59547T, LYQ131, PQ579907, PQ579904, JBJFZU000000000, JBJFZT000000000",JBJFZT000000000,LYQ131,Liu et al. 2026,3378797.0,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079
5,Serinicoccus shuyuelongi,Serinicoccus sp. LYQ92,"LYQ92T, GDMCC 1.5391T, KCTC 59547T, LYQ131, PQ579907, PQ579904, JBJFZU000000000, JBJFZT000000000",JBJFZU000000000,LYQ92,Liu et al. 2026,3378798.0,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079
6,Deinococcus pantiae,Deinococcus sp. SM5_A1,"SM5_A1T, JCM 36669T, KCTC 43670T",JBJGDW000000000,SM5_A1,Jiya et al. 2026,3379094.0,https://doi.org/10.1099/ijsem.0.007071,http://dx.doi.org/10.1099/ijsem.0.007071
7,Halosubterraneus shenae,Haloparvum sp. AD34,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",JBMWNX000000000,AD34,Ding et al. 2026,3234950.0,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
8,Halosubterraneus shenae,Haloparvum sp. PAK95,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",JBNLVO000000000,PAK95,Ding et al. 2026,3418962.0,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
9,Halosubterraneus shenae,Haloparvum sp.,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",OR734406,AD34,Ding et al. 2026,1963352.0,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081


In [17]:
def highlight_rows(row):
    ijsemvalue = row.loc['PublishedName']
    ncbivalue = row.loc['NCBIname']
    if ijsemvalue != ncbivalue:
        color = '#FFB3BA' # Red
    elif ijsemvalue == ncbivalue:
        color = '#BAFFC9' # Green
    return ['background-color: {}'.format(color) for r in row]

new_df = combine_df.style.apply(highlight_rows, axis=1, subset=['PublishedName', 'NCBIname'])
new_df

,PublishedName,NCBIname,Strains,accession,strain,Authority,taxid,DOI,filtered_url
0,Pectobacterium sinaloense,Pectobacterium sinaloense,"LFLA-215T, NCCB 101086T, CMCNRG 1201T, ATCC TSD-577",CP195798,LFLA-215,Valdez-López et al. 2026,3419008.000000,https://doi.org/10.1099/ijsem.0.007076,http://dx.doi.org/10.1099/ijsem.0.007076
1,Halotolerantifilum yawarlongkerapense,Eubacteriales bacterium SD5,"SD5T, DSM 117693T, ATCC TSD-463T",CP199719,SD5,Fisher et al. 2026,3459703.000000,https://doi.org/10.1099/ijsem.0.007084,http://dx.doi.org/10.1099/ijsem.0.007084
2,Ornithinimicrobium jinqii,Ornithinimicrobium sp. LYQ103,"LYQ121T, GDMCC 1.5402T, KCTC 59549T, LYQ103, PQ579903, PQ579902, JBJFZS000000000, JBJFZR000000000",JBJFZR000000000,LYQ103,Liu et al. 2026,3378796.000000,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079
3,Ornithinimicrobium jinqii,Ornithinimicrobium sp. LYQ121,"LYQ121T, GDMCC 1.5402T, KCTC 59549T, LYQ103, PQ579903, PQ579902, JBJFZS000000000, JBJFZR000000000",JBJFZS000000000,LYQ121,Liu et al. 2026,3378801.000000,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079
4,Serinicoccus shuyuelongi,Serinicoccus sp. LYQ131,"LYQ92T, GDMCC 1.5391T, KCTC 59547T, LYQ131, PQ579907, PQ579904, JBJFZU000000000, JBJFZT000000000",JBJFZT000000000,LYQ131,Liu et al. 2026,3378797.000000,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079
5,Serinicoccus shuyuelongi,Serinicoccus sp. LYQ92,"LYQ92T, GDMCC 1.5391T, KCTC 59547T, LYQ131, PQ579907, PQ579904, JBJFZU000000000, JBJFZT000000000",JBJFZU000000000,LYQ92,Liu et al. 2026,3378798.000000,https://doi.org/10.1099/ijsem.0.007079,http://dx.doi.org/10.1099/ijsem.0.007079
6,Deinococcus pantiae,Deinococcus sp. SM5_A1,"SM5_A1T, JCM 36669T, KCTC 43670T",JBJGDW000000000,SM5_A1,Jiya et al. 2026,3379094.000000,https://doi.org/10.1099/ijsem.0.007071,http://dx.doi.org/10.1099/ijsem.0.007071
7,Halosubterraneus shenae,Haloparvum sp. AD34,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",JBMWNX000000000,AD34,Ding et al. 2026,3234950.000000,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
8,Halosubterraneus shenae,Haloparvum sp. PAK95,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",JBNLVO000000000,PAK95,Ding et al. 2026,3418962.000000,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081
9,Halosubterraneus shenae,Haloparvum sp.,"AD34T, MCCC 4K00176T, KCTC 4324T, PAK95, CGMCC 1.62791, MCCC 4K00231, KCTC 4376",OR734406,AD34,Ding et al. 2026,1963352.000000,https://doi.org/10.1099/ijsem.0.007081,http://dx.doi.org/10.1099/ijsem.0.007081


### write output to excel

In [41]:
new_df.to_excel(output, engine='xlsxwriter', index = False, na_rep = '') 

In [ ]:
combine_df.dtypes